# 🤖 Notebook 04 — Introduction au Machine Learning

Le **Machine Learning** permet à un programme d'apprendre à partir de données sans être explicitement programmé.

### 🎯 Objectifs de ce notebook
- Comprendre le pipeline ML standard
- Préparer les données pour le ML
- Entraîner et évaluer des modèles de **classification** et **régression**
- Utiliser la validation croisée
- Interpréter les résultats

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, mean_squared_error, r2_score)

# Modèles de classification
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ Scikit-learn prêt !")

## 1. Le pipeline ML standard

```
Données brutes
     │
     ▼
1️⃣ Préparation des données (nettoyage, encodage, scaling)
     │
     ▼
2️⃣ Split Train / Test
     │
     ▼
3️⃣ Entraînement du modèle (sur Train)
     │
     ▼
4️⃣ Évaluation (sur Test)
     │
     ▼
5️⃣ Optimisation & Validation croisée
```

## 2. Cas 1 : Classification — Prédire la survie du Titanic

In [ ]:
# === ÉTAPE 1 : Chargement et préparation des données ===
titanic = sns.load_dataset('titanic')

# Sélection des features pertinentes
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target   = 'survived'

df = titanic[features + [target]].copy()

# Nettoyage
df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

# Encodage des variables catégorielles
df['sex']      = LabelEncoder().fit_transform(df['sex'])
df['embarked'] = LabelEncoder().fit_transform(df['embarked'])

print(f"Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"Valeurs manquantes : {df.isnull().sum().sum()}")
print(f"\nDistribution cible : {df[target].value_counts().to_dict()}")
df.head()

In [ ]:
# === ÉTAPE 2 : Split Train / Test ===
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} exemples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]} exemples ({X_test.shape[0]/len(X)*100:.0f}%)")

In [ ]:
# === ÉTAPE 3 : Entraînement et comparaison de modèles ===
modeles = {
    'Régression Logistique' : LogisticRegression(max_iter=1000, random_state=42),
    'Arbre de Décision'     : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'         : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM'                   : SVC(random_state=42),
    'K-Nearest Neighbors'   : KNeighborsClassifier(n_neighbors=5)
}

# Scaling (nécessaire pour LR, SVM, KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

resultats = []
for nom, modele in modeles.items():
    modele.fit(X_train_scaled, y_train)
    y_pred = modele.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(modele, X_train_scaled, y_train, cv=5)
    resultats.append({
        'Modèle'     : nom,
        'Accuracy'   : acc,
        'CV mean'    : cv_scores.mean(),
        'CV std'     : cv_scores.std()
    })

df_resultats = pd.DataFrame(resultats).sort_values('Accuracy', ascending=False)
df_resultats.style.format({'Accuracy': '{:.1%}', 'CV mean': '{:.1%}', 'CV std': '{:.3f}'})

In [ ]:
# Visualisation des performances
fig, ax = plt.subplots(figsize=(10, 5))

models_names = df_resultats['Modèle']
accuracies   = df_resultats['Accuracy']
cv_means     = df_resultats['CV mean']
cv_stds      = df_resultats['CV std']

x = np.arange(len(models_names))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy Test', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, cv_means, width, yerr=cv_stds, label='CV moyen ± std',
               color='coral', alpha=0.8, capsize=5)

ax.set_ylabel('Score')
ax.set_title('Comparaison des modèles — Titanic (Survie)')
ax.set_xticks(x)
ax.set_xticklabels(models_names, rotation=20, ha='right')
ax.legend()
ax.set_ylim(0.6, 0.9)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# === ÉTAPE 4 : Analyse détaillée du meilleur modèle ===
meilleur_modele = RandomForestClassifier(n_estimators=100, random_state=42)
meilleur_modele.fit(X_train_scaled, y_train)
y_pred = meilleur_modele.predict(X_test_scaled)

print("=== RAPPORT DE CLASSIFICATION ===")
print(classification_report(y_test, y_pred, target_names=['Décédé', 'Survivant']))

In [ ]:
# Matrice de confusion + Importance des features
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Décédé', 'Survivant'],
            yticklabels=['Décédé', 'Survivant'])
axes[0].set_title('Matrice de confusion')
axes[0].set_ylabel('Réel')
axes[0].set_xlabel('Prédit')

# Importance des features
importances = pd.Series(meilleur_modele.feature_importances_, index=features)
importances.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Importance des variables (Random Forest)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 3. Cas 2 : Régression — Prédire le prix d'une maison

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor

# Chargement du dataset
housing = fetch_california_housing(as_frame=True)
df_housing = housing.frame

print(f"Dataset : {df_housing.shape}")
print(f"Target : Prix médian des maisons (en 100k$)")
df_housing.describe()

In [ ]:
# Préparation et entraînement
X = df_housing.drop('MedHouseVal', axis=1)
y = df_housing['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modeles_reg = {
    'Régression Linéaire'    : LinearRegression(),
    'Random Forest'          : RandomForestClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting'      : GradientBoostingRegressor(n_estimators=100, random_state=42)
}

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"{'Modèle':<25} {'RMSE':>8} {'R²':>8}")
print("-" * 45)

for nom, modele in [('Régression Linéaire', LinearRegression()),
                     ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, random_state=42))]:
    modele.fit(X_train_s, y_train)
    y_pred = modele.predict(X_test_s)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    print(f"{nom:<25} {rmse:>8.3f} {r2:>8.3f}")

In [ ]:
# Visualisation : Prédictions vs Réalité
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train_s, y_train)
y_pred_gb = gb.predict(X_test_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter prédit vs réel
axes[0].scatter(y_test, y_pred_gb, alpha=0.3, color='steelblue', s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('Valeur réelle')
axes[0].set_ylabel('Valeur prédite')
axes[0].set_title(f'Gradient Boosting — Prédits vs Réels (R²={r2_score(y_test, y_pred_gb):.3f})')
axes[0].legend()

# Distribution des résidus
residus = y_test - y_pred_gb
axes[1].hist(residus, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Distribution des résidus')
axes[1].set_xlabel('Résidu (réel - prédit)')

plt.tight_layout()
plt.show()

## 4. Bonnes pratiques et pièges à éviter

### ⚠️ Le surapprentissage (Overfitting)

Un modèle trop complexe "mémorise" les données d'entraînement mais généralise mal.

In [ ]:
# Démonstration overfitting vs underfitting
from sklearn.preprocessing import PolynomialFeatures

X_demo = np.sort(np.random.uniform(-3, 3, 50)).reshape(-1, 1)
y_demo = np.sin(X_demo).ravel() + np.random.normal(0, 0.3, 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titres = ['Sous-apprentissage\n(modèle trop simple)', 'Bon équilibre', 'Sur-apprentissage\n(modèle trop complexe)']
degres = [1, 4, 15]

x_line = np.linspace(-3, 3, 300).reshape(-1, 1)

for ax, d, titre in zip(axes, degres, titres):
    poly = PolynomialFeatures(degree=d)
    X_poly = poly.fit_transform(X_demo)
    model = LinearRegression()
    model.fit(X_poly, y_demo)
    y_line = model.predict(poly.transform(x_line))
    
    ax.scatter(X_demo, y_demo, color='steelblue', alpha=0.6, s=30)
    ax.plot(x_line, y_line, 'r-', linewidth=2)
    ax.set_title(titre)
    ax.set_ylim(-2, 2)

plt.suptitle('Underfitting vs Overfitting', fontsize=13)
plt.tight_layout()
plt.show()

## 5. ✏️ Exercices pratiques

In [ ]:
# EXERCICE 1 — Classification
# Utilisez le dataset Iris pour classifier les espèces de fleurs
# 1. Préparez les données (pas d'encodage nécessaire, les features sont numériques)
# 2. Entraînez un Random Forest
# 3. Affichez l'accuracy et la matrice de confusion

from sklearn.datasets import load_iris
iris_sk = load_iris(as_frame=True)

# Votre code ici


In [ ]:
# EXERCICE 2 — Optimisation
# Utilisez GridSearchCV pour trouver les meilleurs hyperparamètres
# d'un RandomForestClassifier sur le Titanic
# Paramètres à tester : n_estimators=[50, 100, 200], max_depth=[3, 5, 10]

# Votre code ici


---

## ✅ Résumé du notebook 04

| Concept | Ce que vous avez appris |
|---------|-------------------------|
| Pipeline ML | Préparation → Split → Train → Évaluation |
| Classification | Accuracy, F1-score, matrice de confusion |
| Régression | RMSE, R², résidus |
| Validation | Cross-validation, GridSearchCV |
| Pièges | Overfitting, underfitting, data leakage |

## 🎉 Félicitations — Formation terminée !

### 📚 Pour aller plus loin :
- **Deep Learning** : TensorFlow, PyTorch, Keras
- **NLP** : NLTK, spaCy, Transformers (HuggingFace)
- **MLOps** : MLflow, DVC, déploiement de modèles
- **Big Data** : PySpark, Dask

---
*Formation Data Science Python — Notebook 04/04*